In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, RandomSampler
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt
import gc, time

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# TypiClust Selection


class TypiClust:
    def __init__(self, budget_size):
        self.budget_size = budget_size

    def select_samples(self, features):
        kmeans = MiniBatchKMeans(n_clusters=self.budget_size, n_init=3,
                                batch_size=1024, random_state=42)
        labels = kmeans.fit_predict(features)
        nbrs = NearestNeighbors(n_neighbors=21, algorithm='auto', n_jobs=-1)
        nbrs.fit(features)
        dists, _ = nbrs.kneighbors(features)
        density = 1.0 / (dists[:, 1:].mean(axis=1) + 1e-8)
        selected = []
        for c in range(self.budget_size):
            idx = np.where(labels == c)[0]
            best = idx[np.argmax(density[idx])]
            selected.append(int(best))
        return selected



# WideResNet-28-2 (Paper Appendix F.2.3)


class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, activate_before_residual=False):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.relu1 = nn.LeakyReLU(0.1)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=True)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.LeakyReLU(0.1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=True)
        self.activate_before_residual = activate_before_residual
        self.equal = (in_ch == out_ch)
        self.shortcut = None if self.equal else nn.Conv2d(in_ch, out_ch, 1, stride, 0, bias=True)

    def forward(self, x):
        if not self.equal and self.activate_before_residual:
            x = self.relu1(self.bn1(x))
            out = self.conv1(x)
        else:
            out = self.conv1(self.relu1(self.bn1(x)))
        out = self.conv2(self.relu2(self.bn2(out)))
        return out + (x if self.equal else self.shortcut(x))


class WideResNet(nn.Module):
    def __init__(self, num_classes=10, depth=28, widen=2):
        super().__init__()
        ch = [16, 16*widen, 32*widen, 64*widen]
        n = (depth - 4) // 6
        def grp(nb, ic, oc, s, af=False):
            return nn.Sequential(*[BasicBlock(ic if i==0 else oc, oc, s if i==0 else 1,
                   activate_before_residual=(i==0 and af)) for i in range(nb)])
        self.conv1 = nn.Conv2d(3, ch[0], 3, 1, 1, bias=True)
        self.b1 = grp(n, ch[0], ch[1], 1, af=True)
        self.b2 = grp(n, ch[1], ch[2], 2)
        self.b3 = grp(n, ch[2], ch[3], 2)
        self.bn = nn.BatchNorm2d(ch[3])
        self.relu = nn.LeakyReLU(0.1)
        self.fc = nn.Linear(ch[3], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0); nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        return self.fc(F.adaptive_avg_pool2d(
            self.relu(self.bn(self.b3(self.b2(self.b1(self.conv1(x)))))), 1).flatten(1))



# Dataset Wrappers & Augmentations


MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2023, 0.1994, 0.2010)

class LabeledSet(Dataset):
    def __init__(self, dataset, indices, tf):
        self.data = dataset; self.idx = list(indices); self.tf = tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        img, y = self.data[self.idx[i]]; return self.tf(img), y

class UnlabeledSet(Dataset):
    def __init__(self, dataset, indices, wtf, stf):
        self.data = dataset; self.idx = list(indices); self.wtf = wtf; self.stf = stf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        img, _ = self.data[self.idx[i]]; return self.wtf(img), self.stf(img)

class Cutout:
    def __init__(self, s=16): self.s = s
    def __call__(self, img):
        h, w = img.shape[1], img.shape[2]
        cy, cx = np.random.randint(h), np.random.randint(w)
        y1, y2 = max(0,cy-self.s//2), min(h,cy+self.s//2)
        x1, x2 = max(0,cx-self.s//2), min(w,cx+self.s//2)
        img[:, y1:y2, x1:x2] = 0.0; return img

def weak_tf():
    return transforms.Compose([transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(), transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)])

def strong_tf():
    return transforms.Compose([transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(), transforms.RandAugment(num_ops=2, magnitude=10),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD), Cutout(16)])


# Sample Selection
# Ensures all 10 classes are covered

def select_class_balanced(dataset, num_per_class=1):
    """Select 1 random sample per class (10 total for CIFAR-10).
    Paper Section 4.2.3: 'we add a class-balanced random baseline'."""
    targets = np.array(dataset.targets)
    selected = []
    rng = np.random.RandomState(42)
    for c in range(10):
        class_idx = np.where(targets == c)[0]
        chosen = rng.choice(class_idx, size=num_per_class, replace=False)
        selected.extend(chosen.tolist())
    return selected


def ensure_class_coverage(selected_indices, dataset, min_classes=10):
    """If TypiClust selection misses classes, replace worst samples
    to ensure all classes are covered."""
    targets = np.array(dataset.targets)
    selected_labels = targets[selected_indices]
    covered = set(selected_labels)

    if len(covered) >= min_classes:
        return selected_indices  # Already good

    missing = set(range(min_classes)) - covered
    print(f"   TypiClust covers {len(covered)}/10 classes. Missing: {missing}")
    print(f"   Replacing {len(missing)} samples to ensure full coverage")

    result = list(selected_indices)
    rng = np.random.RandomState(42)

    # Find duplicate classes to replace
    from collections import Counter
    label_counts = Counter(selected_labels.tolist())
    replaceable = []
    for cls, count in label_counts.most_common():
        if count > 1:
            cls_positions = [i for i, l in enumerate(selected_labels) if l == cls]
            replaceable.extend(cls_positions[1:])  # Keep first, rest replaceable

    for missing_cls in missing:
        class_idx = np.where(targets == missing_cls)[0]
        new_sample = rng.choice(class_idx)
        if replaceable:
            pos = replaceable.pop(0)
            result[pos] = int(new_sample)
        else:
            result.append(int(new_sample))

    new_labels = targets[result]
    print(f"   New class dist: {dict(zip(*np.unique(new_labels, return_counts=True)))}")
    return result


# FixMatch Training

@torch.no_grad()
def update_ema(model, ema, decay=0.999):
    for ep, mp in zip(ema.parameters(), model.parameters()):
        ep.data.mul_(decay).add_(mp.data, alpha=1.0 - decay)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    model.train(); return 100.0 * correct / total

def train_fixmatch(
    labeled_indices,
    dataset_root="./data",
    total_iterations=50_000,
    batch_size_labeled=32,
    mu=3,
    lr=0.03,
    tau=0.5,
    lambda_u=1.0,
    eval_every=2000,
    label="",
):
    bs_u = mu * batch_size_labeled

    raw = torchvision.datasets.CIFAR10(root=dataset_root, train=True, download=True, transform=None)
    test_ds = torchvision.datasets.CIFAR10(root=dataset_root, train=False, download=True,
        transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)]))

    u_idx = sorted(set(range(len(raw))) - set(labeled_indices))
    labels_arr = [raw.targets[i] for i in labeled_indices]
    unique, counts = np.unique(labels_arr, return_counts=True)
    print(f"\n  [{label}] Labeled: {len(labeled_indices)}, Classes: {len(unique)}/10")
    print(f"  Class dist: {dict(zip(unique.tolist(), counts.tolist()))}")

    l_ds = LabeledSet(raw, labeled_indices, weak_tf())
    u_ds = UnlabeledSet(raw, u_idx, weak_tf(), strong_tf())

    l_loader = DataLoader(l_ds, batch_size=min(batch_size_labeled, len(l_ds)),
        sampler=RandomSampler(l_ds, replacement=True), num_workers=2, drop_last=True)
    u_loader = DataLoader(u_ds, batch_size=min(bs_u, len(u_ds)),
        sampler=RandomSampler(u_ds, replacement=True), num_workers=2, drop_last=True)
    t_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)

    model = WideResNet().to(DEVICE)
    ema = WideResNet().to(DEVICE)
    ema.load_state_dict(model.state_dict())
    for p in ema.parameters(): p.requires_grad_(False)

    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_iterations)

    print(f"  FixMatch: {total_iterations} iters, tau={tau}")

    history = {"steps": [], "acc": [], "best_acc": [],
               "loss_s": [], "loss_u": [], "mask_ratio": []}

    li, ui = iter(l_loader), iter(u_loader)
    best_acc = 0.0; model.train(); t0 = time.time()

    for step in range(1, total_iterations + 1):
        try: x_l, y_l = next(li)
        except StopIteration: li = iter(l_loader); x_l, y_l = next(li)
        try: x_uw, x_us = next(ui)
        except StopIteration: ui = iter(u_loader); x_uw, x_us = next(ui)

        x_l, y_l = x_l.to(DEVICE), y_l.to(DEVICE)
        x_uw, x_us = x_uw.to(DEVICE), x_us.to(DEVICE)

        loss_s = F.cross_entropy(model(x_l), y_l)
        with torch.no_grad():
            probs = torch.softmax(model(x_uw), dim=1)
            max_p, pseudo = probs.max(dim=1)
            mask = max_p.ge(tau).float()
        loss_u = (F.cross_entropy(model(x_us), pseudo, reduction="none") * mask).mean()

        (loss_s + lambda_u * loss_u).backward()
        opt.step(); opt.zero_grad(); sched.step()
        update_ema(model, ema)

        if step % eval_every == 0 or step == 1:
            acc = evaluate(ema, t_loader)
            best_acc = max(best_acc, acc)
            mins = (time.time() - t0) / 60

            history["steps"].append(step)
            history["acc"].append(acc)
            history["best_acc"].append(best_acc)
            history["loss_s"].append(loss_s.item())
            history["loss_u"].append(loss_u.item())
            history["mask_ratio"].append(mask.mean().item())

            print(f"  [{step:6d}/{total_iterations}] Ls={loss_s:.3f} Lu={loss_u:.3f} "
                  f"mask={mask.mean():.2f} acc={acc:.1f}% best={best_acc:.1f}% ({mins:.0f}min)")

    final = evaluate(ema, t_loader)
    best_acc = max(best_acc, final)
    history["steps"].append(total_iterations)
    history["acc"].append(final)
    history["best_acc"].append(best_acc)
    history["loss_s"].append(loss_s.item())
    history["loss_u"].append(loss_u.item())
    history["mask_ratio"].append(mask.mean().item())

    # Free GPU memory
    del model, ema, opt
    torch.cuda.empty_cache(); gc.collect()

    print(f"  Done. Final={final:.1f}%  Best={best_acc:.1f}%")
    return best_acc, history


# MAIN

print("="*50)
print("Step 1: Extracting features for TypiClust...")
print("="*50)
cifar_raw = torchvision.datasets.CIFAR10(root="./data", train=True, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)]))

extractor = torchvision.models.resnet18(weights="IMAGENET1K_V1")
extractor.fc = nn.Identity()
extractor = extractor.to(DEVICE).eval()

features = []
loader = DataLoader(cifar_raw, batch_size=256, shuffle=False, num_workers=0)
with torch.no_grad():
    for x, _ in loader:
        f = F.normalize(extractor(x.to(DEVICE)), dim=1)
        features.append(f.cpu().numpy())
features = np.concatenate(features)
print(f"Features: {features.shape}")

del extractor, loader
torch.cuda.empty_cache(); gc.collect()

# ── Step 2: TypiClust selection + class coverage fix ──
print("\n" + "="*50)
print("Step 2: TypiClust selecting 10 samples...")
print("="*50)
sampler = TypiClust(budget_size=10)
typiclust_indices = sampler.select_samples(features)
del features, sampler; gc.collect()

# Fix class coverage
typiclust_indices = ensure_class_coverage(typiclust_indices, cifar_raw)
print(f"TypiClust indices: {typiclust_indices}")

# Also prepare class-balanced baseline (paper mentions this)
balanced_indices = select_class_balanced(cifar_raw)
print(f"Balanced indices:  {balanced_indices}")

del cifar_raw; gc.collect()

# ── Step 3: Train both ──
print("\n" + "="*50)
print("Step 3: FixMatch Training")
print("="*50)

ITERS = 50_000
EVAL_EVERY = 2000

# Run TypiClust selection
best_typi, hist_typi = train_fixmatch(
    labeled_indices=typiclust_indices,
    total_iterations=ITERS, eval_every=EVAL_EVERY,
    label="TypiClust",
)

# Run class-balanced baseline
best_bal, hist_bal = train_fixmatch(
    labeled_indices=balanced_indices,
    total_iterations=ITERS, eval_every=EVAL_EVERY,
    label="Balanced",
)

print(f"\n{'='*50}")
print(f"  TypiClust best: {best_typi:.2f}%")
print(f"  Balanced  best: {best_bal:.2f}%")
print(f"{'='*50}")


# ==========================================
# 7. Plot Results
# ==========================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"TypiClust + FixMatch on CIFAR-10 (10 labels)",
             fontsize=14, fontweight="bold")

# (1) Accuracy comparison
ax = axes[0, 0]
ax.plot(hist_typi["steps"], hist_typi["acc"], color="#2196F3", linewidth=1.2,
        alpha=0.5, label=f"TypiClust (current)")
ax.plot(hist_typi["steps"], hist_typi["best_acc"], color="#2196F3", linewidth=2,
        label=f"TypiClust best={best_typi:.1f}%")
ax.plot(hist_bal["steps"], hist_bal["acc"], color="#F44336", linewidth=1.2,
        alpha=0.5, label=f"Balanced (current)")
ax.plot(hist_bal["steps"], hist_bal["best_acc"], color="#F44336", linewidth=2,
        linestyle="--", label=f"Balanced best={best_bal:.1f}%")
ax.set_xlabel("Iteration"); ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Test Accuracy: TypiClust vs Balanced"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# (2) Losses (TypiClust)
ax = axes[0, 1]
ax.plot(hist_typi["steps"], hist_typi["loss_s"], color="#FF9800", linewidth=1.2, label="Supervised")
ax.plot(hist_typi["steps"], hist_typi["loss_u"], color="#9C27B0", linewidth=1.2, label="Unsupervised")
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss")
ax.set_title("TypiClust Training Losses"); ax.legend(); ax.grid(True, alpha=0.3)

# (4) Summary
ax = axes[1, 1]; ax.axis("off")
summary = (
    f"Training Summary\n{'─'*40}\n\n"
    f"Dataset:        CIFAR-10\n"
    f"Labels:         10 (1 per class)\n"
    f"Model:          WideResNet-28-2\n"
    f"Iterations:     {ITERS:,}\n"
    f"LR / Tau:       0.03 / 0.95\n"
    f"Batch (L/U):    32 / 96\n\n"
    f"TypiClust best: {best_typi:.2f}%\n"
    f"Balanced  best: {best_bal:.2f}%\n\n"
    f"Paper (400k, mu=7):\n"
    f"  TypiClust+FlexMatch: ~93.2%\n"
    f"  Random:              ~53.8%\n"
    f"  Balanced:            ~80.5%"
)
ax.text(0.08, 0.95, summary, transform=ax.transAxes, fontsize=11.5,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#f5f5f5", alpha=0.9))

plt.tight_layout()
plt.savefig("fixmatch_results.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fixmatch_results.png")

Using device: cuda
Step 1: Extracting features for TypiClust...
Features: (50000, 512)

Step 2: TypiClust selecting 10 samples...
  ⚠ TypiClust covers 4/10 classes. Missing: {2, 3, 4, 5, 7, 8}
  → Replacing 6 samples to ensure full coverage
  → New class dist: {np.int64(0): np.int64(1), np.int64(1): np.int64(1), np.int64(2): np.int64(1), np.int64(3): np.int64(1), np.int64(4): np.int64(1), np.int64(5): np.int64(1), np.int64(6): np.int64(1), np.int64(7): np.int64(1), np.int64(8): np.int64(1), np.int64(9): np.int64(1)}
TypiClust indices: [25317, 11276, 8340, 29627, 9584, 37863, 30875, 44091, 4814, 34468]
Balanced indices:  [14958, 11109, 10940, 20496, 38941, 1133, 15116, 22403, 5643, 5244]

Step 3: FixMatch Training

  [TypiClust] Labeled: 10, Classes: 10/10
  Class dist: {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
  FixMatch: 50000 iters, tau=0.5
  [     1/50000] Ls=2.455 Lu=0.000 mask=0.00 acc=9.0% best=9.0% (0min)
  [  2000/50000] Ls=0.009 Lu=0.583 mask=0.83 acc=10.0% 